# M16 — Verified Baselines & Cross-Domain Validation (Colab)

**Status of this notebook: Phase 1 (task T04) — authored, never executed by its author.** Later sessions add T06 (NFL task targets + reliability + MDE), T09 (baseline learner ladder) and T10 (five-fold retrieval, leakage audit, G16).

## A. Introduction

- **Objective of this phase.** Verify — not rebuild — the official Combine baseline **M1** (`draft_year, height, weight, forty, ten`, from `nflbowl.data.validation.M1_COLS`) on the official target **`nfl.q90_shared`**: recompute the official ladder in a new experiment and require equality at **1e-8** with (1) an independent refit of the same session and (2) every compatible persisted official OOF record of M12 / M13 / M14.
- **Why.** Every later comparison (M16 baselines, M18 ladder B0–B7) is measured against M1; an irreproducible M1 makes them meaningless. A mismatch is a **FAILED_VALIDATION** with athlete-level evidence, never a relaxed tolerance.
- **Two targets, never mixed.** `nfl.q90_shared` (official, T0) and `nfl_speed_q90_median` (exploratory notebook, T0x) are audited separately; their RMSEs are never compared.
- **Inputs.** Bundle (`athletes.parquet` holds M1 + target); read-only M12/M13/M14 experiment folders (`evaluation/records/transfer_oof__*.parquet`); optional exploratory-target file.
- **Outputs.** `tables/baselines/{m1_contract_columns, m1_reproducibility, m1_checks, m1_metrics, m1_athlete_diffs, m1_persisted_records}`, `tables/synthesis/g16b_gate`, `manifests/{m1_contract.json, m1_verification.json}`, `reports/markdown/m1_audit_report.md`, figures `M16_A1_*`.
- **Gate.** G16-B: PASS only when every comparison passes; with no compatible persisted record it stays **NOT_EVALUATED**.

## B. Setup

In [ ]:
# Setup 1 — mount Drive and install nflbowl from GitHub (the Colab PyTorch / CUDA build is never replaced)
from google.colab import drive
drive.mount('/content/drive')
REPO = 'git+https://github.com/mybenkhadda/NFL_bowl.git@main'
%pip install -q {REPO}
%pip install -q --upgrade --force-reinstall --no-deps {REPO}
%pip install -q markdown

In [ ]:
# Setup 2 — version check (after a reinstall: Runtime → Restart session, then continue from here)
import nflbowl
print('nflbowl', nflbowl.__version__)
assert tuple(int(x) for x in nflbowl.__version__.split('.')[:2]) >= (0, 7), 'nflbowl < 0.7.0 is imported: restart the session and rerun from Setup 2'
import json
import polars as pl
from IPython.display import Markdown, display
from nflbowl.lab import handoff
from nflbowl.lab import status as S
from nflbowl.lab.m16 import config as m16cfg, m1_verification as m1v, plots as m16plots, targets as m16targets

In [ ]:
# Setup 3 — configuration (the ONLY cell to edit). M12 / M13 / M14 / M15 directories are read only.
CFG = {
    'experiment_name': 'm16_verified_baselines_v1',
    'bundle_dir': '/content/drive/MyDrive/nfl-big-data-bowl-2027/bundles/m12f_v1_clean',
    'drive_root': '/content/drive/MyDrive/NFLBowl',
    'extra_oof_sources': [],               # further experiment folders holding evaluation/records/transfer_oof__*.parquet
    'exploratory_target_path': '',         # parquet / csv with nfl_id + nfl_speed_q90_median (exploratory notebook export)
    'reproduction_tol': 1e-8,
    'seed': 42,
}
cfg = m16cfg.resolve(CFG)
run = m16cfg.M16Run(cfg.run_dir, cfg)
run.snapshot_config(cfg, handoff.environment(cfg.m13()))
print('run dir                 ', run.root)
print('config fingerprint      ', cfg.fingerprint())
print('persisted-OOF sources   ', [str(p) for p in cfg.experiment_dirs()] or 'none found (G16-B persisted criterion will be NOT_EVALUATED)')
display(run.ledger.table())

In [ ]:
# Setup 4 — the M16 experiment (new directory under RUN_DIR/checkpoints; refuses another bundle or science hash)
def step_open(ctx):
    return handoff.open_experiment(cfg.m13(), run)

EXP = S.run_step(run, run.ledger, 'E16.open', step_open, note='open the M16 experiment on the official bundle / science')
if EXP is None:
    display(run.ledger.table())
    raise SystemExit('the experiment could not be opened — see the reason above and logs/')
print('experiment', EXP.id, '· bundle', EXP.bundle.fingerprint, '· science', EXP.cfg.science_hash())

## C. Data audit — E16.A0 M1 contract · E16.A1 reproducibility (T04)

One guarded step: contract audit → official ladder recomputation (`Experiment.transfer_predictions(models=[])`) → independent refit (`transfer_fairness.baseline_predictions`) → persisted-record discovery and comparison → fold / target checks → tables, audit report, G16-B rows. On a mismatch the step ends FAILED_VALIDATION **after** writing all evidence.

In [ ]:
def step_m1(ctx):
    v = m1v.verify_m1(cfg, run, EXP, ctx)
    ctx['outputs'] += ['tables/baselines/m1_reproducibility', 'manifests/m1_verification.json', 'reports/markdown/m1_audit_report.md']
    return v

M1 = S.run_step(run, run.ledger, 'E16.A1', step_m1, note='official M1 contract + reproducibility', requires=['E16.open'])
summary_path = run.root / 'manifests' / 'm1_verification.json'
SUMMARY = json.loads(summary_path.read_text()) if summary_path.is_file() else None
display(run.ledger.table().filter(pl.col('step') == 'E16.A1'))
if SUMMARY:
    print('VERDICT:', SUMMARY['verdict'], '| internal ok:', SUMMARY['internal_ok'], '| checks ok:', SUMMARY['checks_ok'],
          '| compatible persisted records:', SUMMARY['n_compatible_records'], '| persisted ok:', SUMMARY['persisted_ok'])
    print('failures:', SUMMARY['failures'] or 'none')

In [ ]:
# Evidence tables (re-read from disk, so they also show after a FAILED_VALIDATION)
for name in ('m1_contract_columns', 'm1_reproducibility', 'm1_checks', 'm1_persisted_records', 'm1_metrics'):
    if run.has(name, 'baselines'):
        display(Markdown(f'**{name}**')); display(run.frame(name, 'baselines'))
if run.has('m1_athlete_diffs', 'baselines'):
    d = run.frame('m1_athlete_diffs', 'baselines')
    display(Markdown('**mismatched athletes**')); display(d.filter(pl.col('mismatch')).head(50))

In [ ]:
# Figures (PNG / SVG / PDF + source data)
if run.has('m1_athlete_diffs', 'baselines'):
    fig, dat = m16plots.reproduction_diffs(run.frame('m1_athlete_diffs', 'baselines'), cfg.reproduction_tol)
    run.figure(fig, 'M16_A1_01_m1_reproduction_diffs', data=dat, question='Is the official M1 reproduced athlete by athlete?',
               caption='|Δ prediction| per athlete and comparison (log scale; exact zeros drawn at 1e-17); dashed line = tolerance.')
if run.has('m1_metrics', 'baselines'):
    fig, dat = m16plots.metrics_by_fold(run.frame('m1_metrics', 'baselines'))
    run.figure(fig, 'M16_A1_02_rmse_by_fold', data=dat, question='Do recomputed and persisted baselines agree fold by fold?',
               caption='Out-of-fold RMSE by fold on nfl.q90_shared (descriptive; equality is judged on predictions).')

## D. Exploratory target audit — T0x `nfl_speed_q90_median` (kept separate)

Coverage, distribution and fingerprint of the exploratory target next to the official one. No model is fitted on it here and no RMSE is compared. Without `exploratory_target_path` (and no such column in `athletes.parquet`) the step is **BLOCKED_MISSING_INPUT**.

In [ ]:
def step_t0x(ctx):
    return m1v.exploratory_target_audit(cfg, EXP.bundle.athletes, run)

T0X = S.run_step(run, run.ledger, 'E16.A0-x', step_t0x, note='exploratory target audit (descriptive)', requires=['E16.open'])
if T0X is not None:
    display(run.frame('target_registry_t0_t0x', 'target'))
    print({k: T0X[k] for k in ('overlap_athletes', 'only_T0', 'only_T0x', 'note')})
else:
    display(run.ledger.table().filter(pl.col('step') == 'E16.A0-x'))

## F. Results — G16-B and next experiments

G16-B is read from `tables/synthesis/g16b_gate` (written by the verification). Next: T06 (E16.A2/A3/A6 NFL task targets, reliability, MDE), then T09 (baseline learner ladder) — both reuse the verified M1 frame.

In [ ]:
if run.has('g16b_gate', 'synthesis'):
    display(run.frame('g16b_gate', 'synthesis'))
else:
    print('G16-B: NOT_EVALUATED (the verification did not produce its gate table this session)')
rep = run.root / 'reports' / 'markdown' / 'm1_audit_report.md'
if rep.is_file():
    display(Markdown(rep.read_text()))

## G. Persistence and hand-off

In [ ]:
run.write_status()
run.save_markdown(run.handoff_markdown(), 'm16_handoff')
for rel in ('manifests/m16_status.json', 'manifests/m1_contract.json', 'manifests/m1_verification.json', 'tables/baselines', 'tables/synthesis',
            'figures/png', 'reports/markdown/m1_audit_report.md'):
    p = run.root / rel
    print(('✓ ' if p.exists() else '· ') + str(p))
print('\nSEND BACK: manifests/stages.json, manifests/m16_status.json, manifests/m1_verification.json, logs/*_traceback.txt,')
print('           tables/baselines/m1_reproducibility.csv, tables/baselines/m1_checks.csv, tables/baselines/m1_persisted_records.csv,')
print('           tables/synthesis/g16b_gate.csv, reports/markdown/m1_audit_report.md')